In [1]:
# ==============================================================================
# 1. ENVIRONMENT & DEPENDENCIES
# ==============================================================================
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl evaluate accelerate peft bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-aasuq9l7/unsloth_41a0afc275ea4bb28d866fa8d2a609aa
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-aasuq9l7/unsloth_41a0afc275ea4bb28d866fa8d2a609aa
  Resolved https://github.com/unslothai/unsloth.git to commit cf97faed9ff73af5e46c18a21efdad24ded876dc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 14.1 MB/s eta 0:00:0

In [2]:
import os
import kagglehub
import json
import glob
import gc
import torch
import numpy as np
from datasets import Dataset
from unsloth import FastLanguageModel
from transformers import DataCollatorForLanguageModeling
from trl import SFTTrainer, SFTConfig  

# ==============================================================================
# 1. DOWNLOAD & CLEANUP
# ==============================================================================
dataset_path = kagglehub.dataset_download('adhamashraf202200953/technical-parsed-questions')
print('Dataset Download Complete. Path:', dataset_path)

DATA_FOLDER = "/kaggle/input/datasets/adhamashraf202200953/technical-parsed-questions"
OUTPUT_DIR = "/kaggle/working/outputs/UNSLOTH_MISTRAL_7B"

gc.collect()
torch.cuda.empty_cache()

# ==============================================================================
# 2. DATA LOADING & PARSING
# ==============================================================================
def load_and_parse_custom_jsonl(folder_path):
    all_records = []
    search_pattern = os.path.join(folder_path, "*.jsonl")
    jsonl_files = glob.glob(search_pattern)
    
    for file_path in jsonl_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                try:
                    record = json.loads(line)
                    if all(k in record for k in ["context", "scenario_context", "question", "model_answer"]):
                        all_records.append({
                            "context": record["context"],
                            "scenario_context": record["scenario_context"],
                            "question": record["question"],
                            "model_answer": record["model_answer"]
                        })
                except:
                    continue
    return all_records

raw_parsed_data = load_and_parse_custom_jsonl(DATA_FOLDER)
print(f"Successfully loaded {len(raw_parsed_data)} samples.")
dataset = Dataset.from_list(raw_parsed_data)

# ==============================================================================
# 3. INITIALIZE MODEL & TOKENIZER
# ==============================================================================
max_seq_length = 384 
dtype = None          
load_in_4bit = True   

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "mistralai/Mistral-7B-Instruct-v0.3",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 32,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)


mistral_prompt_style = """<s>[INST] You are an expert technical interviewer. Based on the given textbook context, generate a professional workplace scenario context, an analytical interview question, and a detailed model answer.

Textbook Context:
{CONTEXT} [/INST]
Scenario: {SCENARIO}
Question: {QUESTION}
Model Answer: {ANSWER}</s>"""

def formatting_prompts_func(examples):
    contexts = examples["context"]
    scenarios = examples["scenario_context"]
    questions = examples["question"]
    answers = examples["model_answer"]
    
    texts = []
    for c, s, q, a in zip(contexts, scenarios, questions, answers):
        text = mistral_prompt_style.format(CONTEXT=c, SCENARIO=s, QUESTION=q, ANSWER=a)
        texts.append(text)
        
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=max_seq_length,
        padding=False 
    )
    return tokenized

tokenized_dataset = dataset.map(
    formatting_prompts_func, 
    batched = True, 
    load_from_cache_file = False, 
    remove_columns = ["context", "scenario_context", "question", "model_answer"]
)

split_dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=42)

# ==============================================================================
# 5. ARGUMENTS SETUP (USING SFTConfig)
# ==============================================================================
# 🛠️ FIX: Replaced TrainingArguments with SFTConfig to fix class serialization error.
# Trainer-specific args like max_seq_length and packing are passed here.
training_args = SFTConfig(
    per_device_train_batch_size = 1,  
    per_device_eval_batch_size = 1,   
    gradient_accumulation_steps = 16, 
    warmup_ratio = 0.05,
    num_train_epochs = 3,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    eval_strategy = "steps",
    eval_steps = 25,
    optim = "paged_adamw_8bit", 
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 42,
    output_dir = OUTPUT_DIR,
    report_to = "none",
    save_strategy = "steps",
    save_steps = 25,
    save_total_limit = 1,
    average_tokens_across_devices = False,
    
    # SFT-specific parameters migrated into the config object
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
)

# ==============================================================================
# 6. SAFELY INITIALIZE TRAINER
# ==============================================================================
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = split_dataset["train"],
    eval_dataset = split_dataset["test"],
    args = training_args,
)

# ==============================================================================
# 7. EXECUTE FINE-TUNING & EXPORT
# ==============================================================================
print("Booting Unsloth Training engine with strict caching bypassed...")
trainer.train()

# Save final adapters safely
final_adapter_path = "/kaggle/working/UNSLOTH_MISTRAL_7B_LORA"
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"LoRA adapters successfully exported to: {final_adapter_path}")

# Zip output artifacts
os.system(f'zip -r "/kaggle/working/UNSLOTH-MISTRAL-7B-LORA.zip" "{final_adapter_path}"')
print("Artifact zipped and ready for export.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Dataset Download Complete. Path: /kaggle/input/datasets/adhamashraf202200953/technical-parsed-questions
Successfully loaded 2190 samples.
==((====))==  Unsloth 2026.6.1: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Step,Training Loss,Validation Loss
25,1.434260,1.441000
50,1.259258,1.296337
75,1.287939,1.265232
100,1.215018,1.236833
125,1.252610,1.222229
150,1.130907,1.218547
175,1.112083,1.213030
200,1.079380,1.208126
225,1.118493,1.201692
250,1.141375,1.197867


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_MISTRAL_7B/checkpoint-25/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/outputs/UNSLOTH_MISTRAL_7B/checkpoint-25.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_MISTRAL_7B/checkpoint-50/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/outputs/UNSLOTH_MISTRAL_7B/checkpoint-50.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_MISTRAL_7B/checkpoint-75/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/outputs/UNSLOTH_MISTRAL_7B/checkpoint-75.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_MISTRAL_7B/checkpoint-100/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/outputs/UNSLOTH_MISTRAL_7B/checkpoint-100.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/UNSLOTH_MISTRAL_7B/checkpoint-125/tokenizer_config.json.
U

LoRA adapters successfully exported to: /kaggle/working/UNSLOTH_MISTRAL_7B_LORA
  adding: kaggle/working/UNSLOTH_MISTRAL_7B_LORA/ (stored 0%)
  adding: kaggle/working/UNSLOTH_MISTRAL_7B_LORA/README.md (deflated 65%)
  adding: kaggle/working/UNSLOTH_MISTRAL_7B_LORA/adapter_model.safetensors (deflated 8%)
  adding: kaggle/working/UNSLOTH_MISTRAL_7B_LORA/tokenizer_config.json (deflated 96%)
  adding: kaggle/working/UNSLOTH_MISTRAL_7B_LORA/adapter_config.json (deflated 57%)
  adding: kaggle/working/UNSLOTH_MISTRAL_7B_LORA/tokenizer.model (deflated 61%)
  adding: kaggle/working/UNSLOTH_MISTRAL_7B_LORA/chat_template.jinja (deflated 74%)
  adding: kaggle/working/UNSLOTH_MISTRAL_7B_LORA/tokenizer.json (deflated 85%)
Artifact zipped and ready for export.
